# Batched factual recall (layer × head)

**llm-scalpel** · `BatchExperimentRunner` · multi-prompt activation patching

This notebook runs a **batched** path-patching experiment: several clean/corrupt **city & landmark** pairs in one forward pass per patch site, then aggregates logit-difference effects **across the batch** to see which attention heads *consistently* support factual recall.

## 1. Environment and imports

We use `BatchExperimentRunner` (left-padded batch tokenization via `get_left_padded_tokens` inside the runner) and `causal_patcher.viz.plot_heatmap` for figures. Add the repo root to `sys.path` if needed, or `pip install -e ".[demo]"` from the project root.

**If the *heavy imports* cell seems stuck:** it is not `import sys` — the cell above only adjusts `sys.path`. The imports cell loads `torch`, `transformer_lens`, and the rest; each can take *minutes* on the first run on Windows when the `.venv` is under OneDrive or antivirus scans many DLLs. That cell prints progress; if it never ends, exclude this project’s `.venv` in Windows Security, or put the repo on a local drive (e.g. `C:\dev\llm-scalpel`).

In [ ]:
import sys
from pathlib import Path


In [ ]:

_here = Path.cwd().resolve()
for candidate in (_here, _here.parent, _here.parent.parent):
    if (candidate / "causal_patcher").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
print("sys.path ready (repo root on path).", flush=True)

In [ ]:
# This cell can take *minutes* on first run (PyTorch + TransformerLens + matplotlib + causal_patcher).
# Run the cell above first; if this one stalls, see the note in the Environment section — OneDrive on .venv is a common cause.
print("importing numpy, torch, matplotlib…", flush=True)
import numpy as np
import torch
import matplotlib.pyplot as plt
print("importing transformer_lens (often the slowest)…", flush=True)
from transformer_lens import HookedTransformer
print("importing causal_patcher…", flush=True)
from causal_patcher.batch_runner import BatchExperimentRunner
from causal_patcher.utils import get_left_padded_tokens
from causal_patcher.viz import plot_heatmap

%matplotlib inline
plt.style.use(
    "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default"
)
print("imports done.", flush=True)

## 2. Data: city / landmark pairs

Each example is **(clean prompt, corrupt prompt, clean answer, corrupt answer)**. Answers should be **single BPE tokens** in GPT-2 (we use leading space for city names, e.g. `" Paris"`, as in the single-example notebook). Prompts are short factual frames so the model predicts the city at the end.

In [ ]:
def _as_pairs():
    # (clean_prompt, corrupt_prompt, clean_answer, corrupt_answer)
    # Answer strings must be a *single* GPT-2 BPE token (check with model.to_single_token).
    return [
        ("The Eiffel Tower is in", "The Colosseum is in", " Paris", " Rome"),
        ("The Great Wall is in", "The Acropolis is in", " Beijing", " Athens"),
        ("The Statue of Liberty is in", "Big Ben is in", " New", " London"),
        ("The Sydney Opera House is in", "The Brandenburg Gate is in", " Sydney", " Berlin"),
        ("The Taj Mahal is in", "The Pyramids of Giza are in", " Delhi", " Cairo"),
        ("Niagara Falls borders", "The Grand Canyon is in", " New", " Los"),
        ("The Parthenon is in", "The Louvre is in", " Athens", " Paris"),
    ]


rows = _as_pairs()
clean_prompts = [a for a, _, _, _ in rows]
corrupt_prompts = [b for _, b, _, _ in rows]
clean_answers = [c for _, _, c, _ in rows]
corrupt_answers = [d for _, _, _, d in rows]

print(f"Batch size: {len(clean_prompts)} examples")

In [ ]:
# Optional: show left-padded token shapes (same helper the runner uses internally)
_device = "cuda" if torch.cuda.is_available() else "cpu"
_m = HookedTransformer.from_pretrained("gpt2-small", device=_device)

for name, batch in (
    ("clean", clean_prompts),
    ("corrupt", corrupt_prompts),
):
    toks, mask = get_left_padded_tokens(_m, batch)
    print(f"{name}: tokens {tuple(toks.shape)}, attention_mask {tuple(mask.shape)}")

# Verify each answer string is a single token (runner requirement)
for i, (ca, ua) in enumerate(zip(clean_answers, corrupt_answers)):
    for lab, s in ("clean", ca), ("corrupt", ua):
        _m.to_single_token(s)
print("All answer strings are single-token in GPT-2.")
del _m, _device

## 3. Run baselines and `attn_head_z` patch sweep

We call `run_baselines()` once to cache **clean** and **corrupt** activations, then `run_patch_sweep("attn_head_z", pos=-1)` so each cell is the **logit-difference** readout (clean vs corrupt answer) on the **patched corrupt** run, at the **last** time step, for the full batch.

> **Note:** The sweep runs one corrupt forward per (layer, head) — a few minutes on CPU for `gpt2-small` (12×12 sites).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = HookedTransformer.from_pretrained("gpt2-small", device=device)
torch.manual_seed(0)

runner = BatchExperimentRunner(
    model,
    clean_prompts= clean_prompts,
    corrupt_prompts=corrupt_prompts,
    clean_answers=clean_answers,
    corrupt_answers=corrupt_answers,
)
runner.run_baselines(names_filter=None)  # cache all hooks for the sweep
results = runner.run_patch_sweep("attn_head_z", pos=-1)

print("results shape [n_layers, n_heads, batch_size]:", tuple(results.shape))
assert results.dim() == 3

## 4. Aggregation: mean and std across the batch

We reduce the last dimension (independent **prompts**) to summarize effect sizes per **(layer, head)**.

In [ ]:
mean_results = results.float().mean(dim=-1)
std_results = results.float().std(dim=-1, unbiased=False)

print("mean_results:", tuple(mean_results.shape))
print(" std_results:", tuple(std_results.shape))

mean_np = mean_results.detach().cpu().numpy()
std_np = std_results.detach().cpu().numpy()

## 5. Interpretation (read with the heatmaps below)

With the default `RdBu_r` scale centered at zero, **red / warm** cells mean that patching **increased** the batch-mean logit **difference** (logit of the clean city minus logit of the corrupt city) on the **patched corrupt** forward. Across **different** factual frames, a head that is **red** in the **mean** map is one that **consistently** contributes to recovering the clean answer: it broadly pushes the readout toward the true city rather than the foil.

The **standard-deviation** map highlights **inconsistency**: high **std** means the head’s effect **varies a lot** across prompts (model-internal circuits that are *not* doing the same thing in every example). Use it together with the mean: strong mean + low std is the cleanest “shared factual recall” signal; high std flags fragile or entangled behavior for this set of cities.

In [ ]:
n_layers, n_heads = mean_np.shape
fig, ax, im = plot_heatmap(
    mean_np,
    xlabel="Head index",
    ylabel="Layer",
    title=(
        "Logit diff (patched corrupt): mean over batch | attn_head_z | last position\n"
        f"(batch_size={len(clean_prompts)} city/landmark pairs; RdBu: red = higher clean−corrupt log-odds)"
    ),
    figsize=(9.0, 5.0),
)
ax.set_yticks(np.arange(n_layers))
ax.set_xticks(np.arange(n_heads))
plt.show()

In [ ]:
fig2, ax2, im2 = plot_heatmap(
    std_np,
    xlabel="Head index",
    ylabel="Layer",
    title=(
        "Same metric: standard deviation across batch (per layer, per head)\n"
        f"(higher = more variable effect across the {len(clean_prompts)} prompts)"
    ),
    figsize=(9.0, 5.0),
    cmap="magma",
    center_zero=False,
)
ax2.set_yticks(np.arange(n_layers))
ax2.set_xticks(np.arange(n_heads))
plt.show()